# Capa gold: accidentes en carreteras

Modelo estrella a partir de silver.

- **Entrada:** `data/silver/accidentes.parquet`
- **Salida:** `data/gold/`, un Parquet por tabla
- **Grano del hecho:** una fila por accidente

## Configuración

In [45]:
import pandas as pd
from pathlib import Path

origen  = Path("../data/silver/accidentes.parquet")
destino = Path("../data/gold")

## Lectura

In [46]:
df = pd.read_parquet(origen)
filas_silver = len(df)
df.shape

(60531, 23)

## Crear dimensiones

### Función para crear dimensiones

In [47]:
def crear_dimension(df, columnas, llave):
    dim = df[columnas].drop_duplicates().sort_values(columnas).reset_index(drop=True)
    dim.insert(0, llave, dim.index + 1)
    return dim

### Dimensiones

In [48]:
# Dimensión Causas
dim_causa = crear_dimension(df, ["causa_accidente"], "causa_key")
dim_causa.head()

,causa_key,causa_accidente
0,1,Acceder a la vía sin observar la presencia de ...
1,2,Acceso irregular
2,3,Acumulación de aceite en el suelo
3,4,Acumulación de arena o escombros en el pavimento
4,5,Adelantamiento indebido


In [49]:
# Dimensión Ubicación
dim_ubicacion = crear_dimension(df, ["estado_sigla", "estado"], "ubicacion_key")
dim_ubicacion.head()

,ubicacion_key,estado_sigla,estado
0,1,ES,Espírito Santo
1,2,MG,Minas Gerais
2,3,RJ,Río de Janeiro
3,4,SP,São Paulo


In [50]:
# Dimensión Caracteristicas
dim_caracteristicas = crear_dimension(df, ["clasificacion_accidente", "tipo_carril"], "caracteristicas_key")
dim_caracteristicas.head()

,caracteristicas_key,clasificacion_accidente,tipo_carril
0,1,Con víctimas fatales,Doble
1,2,Con víctimas fatales,Múltiple
2,3,Con víctimas fatales,Simple
3,4,Con víctimas heridas,Doble
4,5,Con víctimas heridas,Múltiple


In [51]:
# Dimensión Calendario
inicio = f"{df['fecha'].min().year}-01-01"
fin    = f"{df['fecha'].max().year}-12-31"

dim_calendario = pd.DataFrame({"fecha": pd.date_range(inicio, fin, freq="D")})

meses = {1: "Ene", 2: "Feb", 3: "Mar", 4: "Abr", 5: "May", 6: "Jun", 7: "Jul", 8: "Ago", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dic"}
dias  = {1: "Lun", 2: "Mar", 3: "Mié", 4: "Jue", 5: "Vie", 6: "Sáb", 7: "Dom"}

dim_calendario["anio"] = dim_calendario["fecha"].dt.year
dim_calendario["trimestre"] = dim_calendario["fecha"].dt.quarter
dim_calendario["mes"] = dim_calendario["fecha"].dt.month
dim_calendario["nombre_mes"] = dim_calendario["mes"].map(meses)
dim_calendario["dia_semana"] = dim_calendario["fecha"].dt.dayofweek + 1
dim_calendario["nombre_dia_semana"] = dim_calendario["dia_semana"].map(dias)

dim_calendario["fecha"] = dim_calendario["fecha"].dt.date
dim_calendario.head()

,fecha,anio,trimestre,mes,nombre_mes,dia_semana,nombre_dia_semana
0,2018-01-01,2018,1,1,Ene,1,Lun
1,2018-01-02,2018,1,1,Ene,2,Mar
2,2018-01-03,2018,1,1,Ene,3,Mié
3,2018-01-04,2018,1,1,Ene,4,Jue
4,2018-01-05,2018,1,1,Ene,5,Vie


In [52]:
# Dimensión Hora
dim_hora = pd.DataFrame({"hora": range(24)})
dim_hora["etiqueta"] = dim_hora["hora"].astype(str).str.zfill(2) + ":00"
dim_hora

,hora,etiqueta
0,0,00:00
1,1,01:00
2,2,02:00
3,3,03:00
4,4,04:00
5,5,05:00
6,6,06:00
7,7,07:00
8,8,08:00
9,9,09:00


## Tabla de hechos

In [53]:
#Hechos Accidentes
fact_accidentes = df.copy()

# Extrae solo la hora
fact_accidentes["hora"] = fact_accidentes["hora"].map(lambda h: h.hour)

# Join para obtener las keys
fact_accidentes = (
    fact_accidentes
    .merge(dim_causa, on="causa_accidente", how="left")
    .merge(dim_caracteristicas, on=["clasificacion_accidente", "tipo_carril"], how="left")
    .merge(dim_ubicacion, on="estado_sigla", how="left")
)

#Renombra columna ID
fact_accidentes = fact_accidentes.rename(columns={"id": "accidente_id"})

fact_accidentes = fact_accidentes[[
    "accidente_id",
    "fecha",
    "hora",
    "ubicacion_key",
    "causa_key",
    "caracteristicas_key",
    "muertos",
    "heridos",
    "latitud",
    "longitud",
]]

fact_accidentes.head()

,accidente_id,fecha,hora,ubicacion_key,causa_key,caracteristicas_key,muertos,heridos,latitud,longitud
0,262002,2020-01-10,21,4,23,4,0,1,-22.531582,-44.753747
1,264173,2020-01-21,8,4,23,4,0,1,-23.260721,-45.951792
2,270555,2020-02-20,21,4,23,4,0,1,-23.366930,-46.230812
3,271883,2020-02-26,17,4,23,4,0,1,-22.774546,-45.138071
4,274215,2020-03-09,6,4,23,4,0,1,-23.492329,-46.551282
